# Super Arigato Shop



## What is Headless Commerce?

separating your store’s front-end from the e-commerce platform to better customize the user experience for your customers

## chat bot integration

* Download csv from shop
* generate llm.txt
* put it on watson x assistant

In [1]:
# routine to generate knowledge llm txt from csv
import pandas as pd

# Load your CSV
df = pd.read_csv('products_knowledge_base.csv')

# Initialize a list to hold the text lines
lines = []

# Iterate through rows to create Q&A or statements
# Assuming columns 'Question' and 'Answer' exist
for index, row in df.iterrows():
    q = row['Question']
    a = row['Answer']
    lines.append(f"Q: {q}\nA: {a}\n")

# If your CSV is just raw data (e.g., Product, Price), format it manually:
# for index, row in df.iterrows():
#     lines.append(f"The price of {row['Product']} is {row['Price']}.\n")

# Write to TXT file
with open('knowledge_base.txt', 'w', encoding='utf-8') as f:
    f.writelines(lines)

print("Conversion complete: knowledge_base.txt created.")   

Conversion complete: knowledge_base.txt created.


In [ ]:
# fast api example middleware to link watson assistant x to knowledge base

# on watson ai assistnt use ${search_result.status_code}
from fastapi import FastAPI

app = FastAPI()

@app.get("/product/{handle}")
async def get_product(handle: str):
    # Query Shopify here
    return {
        "name": "Gojo Figure",
        "price": "39.99",
        "available": True
    }

## how to shopify?
https://shopify.dev/docs/storefronts/headless/building-with-the-storefront-api/getting-started

Creating app is required:
https://dev.shopify.com/dashboard/220771485/apps/375462658049 

Creating a store front api (from shop dashboard) is really important:
https://admin.shopify.com/store/superarigatodev/headless 
This is where token will be retrieved

Cart docs:
https://shopify.dev/docs/api/storefront/latest/mutations/cartLinesAdd 

## ButterCMS

Its important to use this

In [ ]:
## this steps to extract google drive files free

#Set Permissions: Right-click the image in Google Drive, select Share, and change access from "Restricted" to Anyone with the link. 
#Generate Direct Link: Copy the sharing link (format: https://drive.google.com/file/d/FILE_ID/view?usp=sharing). Extract the FILE_ID and prepend it with https://lh3.googleusercontent.com/d/. 



## SHARI

A good guideline that might help answer the question, "Do I need NgRx Store?" is the SHARI principle:

Shared: state that is accessed by many components and services.

Hydrated: state that is persisted and rehydrated from external storage.

Available: state that needs to be available when re-entering routes.

Retrieved: state that must be retrieved with a side-effect.

Impacted: state that is impacted by actions from other sources.

CART USAGE:
Component
    ↓
cartFacade.addLineToCart(...)
    ↓
Dispatch Action
    ↓
Effect
    ↓
Shopify API
    ↓
Success Action
    ↓
Reducer updates store

### CHAT GPT SUGGESTION

What I would do

If you're still early in the project:

superarigato.com
    → Angular

checkout.superarigato.com
    → Shopify

In [6]:
import os

def buscar_disponibilidad_aproximada(handle_recibido: str) -> str:
    file_path = "knowledge_base.txt"
    
    if not os.path.exists(file_path):
        return "No"
        
    # 1. Limpiar el handle y extraer palabras clave de más de 3 letras
    # Ejemplo: "tasa-mario-verde" -> ["tasa", "mario", "verde"]
    palabras_handle = [
        p.lower() for p in handle_recibido.replace("-", " ").split() 
        if len(p) > 3
    ]
    
    # Si el handle no tiene palabras significativas, no podemos buscar de forma segura
    if not palabras_handle:
        return "No"

    # 2. Leer e iterar el archivo línea por línea
    with open(file_path, "r", encoding="utf-8") as f:
        lineas = f.readlines()

    for i, linea in enumerate(lineas):
        linea_lower = linea.lower()
        
        # 3. Filtrar solo las líneas que pregunten por disponibilidad
        if "is" in linea_lower and "available" in linea_lower:
            # Extraer palabras de la línea de más de 3 letras
            palabras_linea = [p for p in linea_lower.split() if len(p) > 3]
            
            # 4. Verificar si ALGUNA palabra del handle coincide con la línea
            # Ejemplo: Si el handle tiene "mario" y la línea tiene "mario", hay coincidencia
            coincidencia = any(palabra in palabras_linea for palabra in palabras_handle)
            
            if coincidencia:
                # 5. Si coincide la pregunta (Q:), revisamos la respuesta (A:) que está en la siguiente línea (i + 1)
                if i + 1 < len(lineas):
                    siguiente_linea = lineas[i + 1].lower()
                    # Validamos si la respuesta confirma la compra exitosa
                    if "currently available" in siguiente_linea and "archived" not in siguiente_linea:
                        return "Yes"
                    else:
                        return "No"

    return "No"

print(buscar_disponibilidad_aproximada("bowser"))


Yes
